<a href="https://colab.research.google.com/github/NicholasAtt/AmazonPrisma/blob/main/notebooks/ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Setup and Data Loading

This section prepares the environment by installing all necessary libraries and loading the dataset from Google Drive.

In [ ]:
print("Starting Setup: Installing required libraries...")
# Install libraries for NLP, Machine Learning, and Visualization
!pip install pandas numpy torch scikit-learn --quiet
!pip install "spacy[transformers]" sentence-transformers --quiet
!pip install umap-learn hdbscan plotly wordcloud seaborn --quiet

# Download a powerful spaCy transformer model for English NER
!python -m spacy download en_core_web_trf --quiet

!pip install spacy-transformers

print("Libraries and NER model installed.")

# Import all the necessary libraries for data handling, NLP, clustering, and plotting
import pandas as pd
import numpy as np
import re
import time
from collections import Counter, defaultdict
import warnings
import os
import gc

import torch
import spacy
from spacy.language import Language

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import umap
import hdbscan

import plotly.express as px
import plotly.graph_objects as go
from wordcloud import WordCloud
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive, auth
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.client import GoogleCredentials

# Set up configurations for pandas and suppress common warnings
warnings.filterwarnings('ignore')
pd.options.display.max_colwidth = 200

# Authenticate with your Google account to access files in Google Drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive_service = GoogleDrive(gauth)

# The ID of the dataset file in Google Drive
id_train = '1kddgLOIVkn5HJw8Ib6rrUh5RkcMwsWeM'
file1 = drive_service.CreateFile({'id': id_train})
# Download the file and load it into a pandas DataFrame
file1.GetContentFile('train.csv.gz')
df_train = pd.read_csv('train.csv.gz', header=None, names=['polarity', 'title', 'text'])

# Select a random sample of 10,000 reviews to make processing faster
SAMPLE_SIZE = 10000
df = df_train.sample(n=SAMPLE_SIZE, random_state=42)

display(df.head())


### 2. Named Entity Recognition (NER) and Data Preparation

This section defines functions to clean the text, extract entities using spaCy, and filter out common, non-specific words. We combine the title and text of each review for a more comprehensive analysis.

In [ ]:
# Function to clean and normalize text for NER
def preprocess_ner(text):
    if not text or pd.isna(text):
        return ""
    # Remove non-ASCII characters and other symbols
    text = text.encode("ascii", "ignore").decode()
    text = re.sub(r'[^\w\s\.\-\'&]', '', text)
    text = re.sub(r'\s+', ' ', text.strip())
    return text

# Combine the 'title' and 'text' columns into a single string for each review
df['combined_text_ner'] = (df['title'].astype(str).apply(preprocess_ner) + ". " +
                           df['text'].astype(str).apply(preprocess_ner))
# Remove any reviews that are too short to contain meaningful entities
df = df[df['combined_text_ner'].str.len() > 20].reset_index(drop=True)

# Check for GPU availability to optimize spaCy's performance
if spacy.prefer_gpu():
    print("GPU detected, spaCy will use it.")
else:
    print("No GPU detected, spaCy will use the CPU")

# Load the powerful spaCy transformer model for NER
ner_model = spacy.load("en_core_web_trf")
print("NER model (spaCy en_core_web_trf) loaded.")

# Function to further clean the extracted entity text (e.g., remove leading/trailing punctuation)
def clean_entity(text):
    text = re.sub(r"^[\W]+|[\W]+$", "", text.strip())
    return text.lower()

# Function to extract entities from a list of texts using the spaCy model
def extract_entities(
    texts_list,
    model,
    batch_size=64,
    # Define the set of relevant entity labels to extract. 'PRODUCT' and 'WORK_OF_ART' are key for reviews.
    relevant_labels={'PERSON', 'ORG', 'GPE', 'PRODUCT', 'WORK_OF_ART', 'EVENT', 'LOC'},
):
    all_entities = []
    print(f"\nExtracting entities from {len(texts_list)} texts...")
    if isinstance(model, Language):
        # Use model.pipe for efficient batch processing
        for doc in model.pipe(texts_list, batch_size=batch_size):
            extracted = set()
            for ent in doc.ents:
                # Filter by relevant labels and a minimum length to avoid single-letter entities
                if ent.label_ in relevant_labels and len(ent.text.strip()) > 1:
                    ent_text = clean_entity(ent.text)
                    extracted.add(ent_text)
            all_entities.append(list(extracted))
    else:
        raise TypeError("The provided model is not a valid spaCy model.")
    print("Entity extraction completed.")
    return all_entities

# Apply the entity extraction function to the combined text data
texts_to_process = df['combined_text_ner'].tolist()
df['extracted_entities'] = extract_entities(texts_to_process, ner_model)

# Define a list of common words that we want to ignore as they are not specific entities
common_words_to_ignore = {'book', 'movie', 'cd', 'product', 'item', 'one', 'review', 'it', 'dvd', 'amazon', 'kindle', 'bible'}

# Function to filter the extracted entities based on the ignore list
def filter_entities(entity_list):
    if not isinstance(entity_list, list):
        return []
    return [entity for entity in entity_list if len(entity) > 2 and entity not in common_words_to_ignore]

# Create new columns for the filtered entities and a single string of entities for later use
df['entities'] = df['extracted_entities'].apply(filter_entities)
df['entity_text'] = df['entities'].apply(lambda ents: ' '.join(ents))

print("\nExamples of entities extracted with the new model:")
display(df[['combined_text_ner', 'entities']].head(10).style.set_properties(**{'text-align': 'left'}))

GPU detected, spaCy will use it.
NER model (spaCy en_core_web_trf) loaded.

Extracting entities from 10000 texts...
Entity extraction completed.

Examples of entities extracted with the new model:


,combined_text_ner,entities
0,Expensive Junk. This product consists of a piece of thin flexible insulating material adhesive backed velcro and white electrical tape.Problems1. Instructions are three pictures with little more information.2. Velcro was all crumpled as received and was stronger than the adhesive. When i tried to disengage the velcro both pieces came off and the paint from the ceiling.3. White electrical tape was horrible... cheap narrow and it fell off in less than 1 hour.4. The price is a ripoff.I am building my own which is easier to use cheaper more attractive and higher r-value. I am surprised Amazon even lists this junk.,[]
1,Toast too dark. Even on the lowest setting the toast is too dark for my liking. Also the on light stays lit so you have to unplug it to avoid wasting electricity. Not the quality I expected from Cuisinart.,['cuisinart']
2,Excellent imagery...dumbed down story. I enjoyed this disc. The video is stunning. I agree with others that the story is very dumbed down and takes a childish approach. It actually seems like its a little one sided and VERY pro-environmental. Nevertheless its enjoyable. I would say however that the Amazon WMV HD disc is a better story and has better and sharper images and more interesting things to look at.,['wmv hd']
3,Are we pretending everyone is married. The authors pretend that parents neither die nor divorce. Insisting the marriage is the rock upon which all else behavior well-being of the child is built they send a clear message to non-traditional households this book is for people who play the game of life our way only and everyone else can suffer the bad behavior they deserve to.,[]
4,Not worth your time. Might as well just use a knife this product holds next to nothing. I found it to be a useless product.,[]
5,Book reads like written for grade schoolers. I think that the whole aspect of hitting the road and going on tour with a band is just about the coolest thing that a person can do. Thus this book seemed like the perfect read for someone like me. But when I got the book I was dissapointed in the fact that it seems like the author took five or six stories from the 'The Babysiters Club' or some other book for young kids rewrote them to include drug and music references and than sold it. The writing is just not very good and the stories are all lame. A girl's mother realizes her 16 year old daughter is right about being on tour a girl runs from the cops and gets away a ticket is fake ect. Those in themselves aren't even that bad but the way he writes makes it a boring read.,"[""the 'the babysiters club""]"
6,Jeanne de Florette & Manon of the Springs. I saw these movies originally in the theater loved both and ordered them because I wanted to have it in my collection. I ordered it twice and neither of the DVDs worked. I would think you would check your merchandise before sending it out. I am so disappointed and can't believe that you could send out two defective DVDs and had not solved the problem after the first one plus some of the complaints I have read in the reviews. I am rating this with one star because of the quality of the DVD not the films.Colleen,"['colleen', 'jeanne de florette']"
7,Theater Projector Ceiling Mount. Would not fit my new Optima HD20 projector -- as projectors get smaller the mounting holes get closer together -- this mount had four arms and the projector had three -- the mount had arms that would not get short enough to reach the three projector holes -- for larger projectors may work better but for mine would not recommend -- returning it was very easy and the company representatives very helpful -- so if you get it and it doesn't work the company is easy to work with --,['optima hd20']
8,This import is sooooooooooo good. This is a great cd It is the normal oops cd plus three cool new songs. Also there are lots of added extras that are inhansed on the cd for example you get the lucky video a fact file and a britney unscrambler The t

### 3. Entity-Based Clustering

Here, we use a classic clustering pipeline: **TF-IDF** to vectorize the entities, **UMAP** for dimensionality reduction, and **HDBSCAN** to find dense clusters. The goal is to discover thematic groups within the reviews based on the entities they discuss.

In [ ]:
print("\nInitializing entity-based clustering pipeline.")

# Filter out reviews that have no entities to avoid errors in the clustering pipeline
non_empty_indices = df['entity_text'].str.strip().astype(bool)
df_filtered = df[non_empty_indices].copy()
if df_filtered.empty:
    raise ValueError("No entities were extracted. Cannot proceed with clustering.")

print(f"\nTF-IDF vectorization of entities (Valid documents: {len(df_filtered)})...\n")
# Use TF-IDF to convert the entity text into numerical vectors, representing the importance of each entity
vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.9,
    max_features=10000,
    ngram_range=(1, 2)
)
X_entities = vectorizer.fit_transform(df_filtered['entity_text'])

print(f"\nDimensionality reduction with UMAP (Entity vocabulary: {X_entities.shape[1]})...\n")
# Apply UMAP to reduce the high-dimensional TF-IDF vectors to 2D for visualization and clustering
umap_reducer = umap.UMAP(n_neighbors=40, min_dist=0.1, n_components=2, metric='cosine', random_state=42)
embedding = umap_reducer.fit_transform(X_entities)
df_filtered['x'] = embedding[:, 0]
df_filtered['y'] = embedding[:, 1]

print("\nRunning clustering with HDBSCAN...\n")
# Apply HDBSCAN, a powerful density-based algorithm that doesn't require pre-defining the number of clusters
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=5, metric='euclidean')
clusterer.fit(embedding)
df_filtered['cluster_label'] = clusterer.labels_

# Report on the clustering results: number of clusters found and noise points
n_clusters = len(set(clusterer.labels_)) - (1 if -1 in clusterer.labels_ else 0)
noise_points = np.sum(clusterer.labels_ == -1)
print(f"Clustering completed: {n_clusters} clusters and {noise_points} noise points.")

# Clean up memory after large computations
del X_entities, embedding
gc.collect()

print("\nCalculating performance metrics...")
# Calculate standard clustering metrics (Silhouette, Calinski-Harabasz, Davies-Bouldin)
clustered_points_df = df_filtered[df_filtered['cluster_label'] != -1]
if n_clusters > 1:
    X_metrics = clustered_points_df[['x', 'y']].values
    labels_metrics = clustered_points_df['cluster_label'].values
    print(f"Silhouette Score: {silhouette_score(X_metrics, labels_metrics):.3f}")
    print(f"Calinski-Harabasz Index: {calinski_harabasz_score(X_metrics, labels_metrics):.3f}")
    print(f"Davies-Bouldin Index: {davies_bouldin_score(X_metrics, labels_metrics):.3f}")
else:
    print("Cannot calculate metrics (at least 2 clusters required).")


Initializing entity-based clustering pipeline.

TF-IDF vectorization of entities (Valid documents: 6341)...


Dimensionality reduction with UMAP (Entity vocabulary: 1187)...


Running clustering with HDBSCAN...

Clustering completed: 201 clusters and 1451 noise points.

Calculating performance metrics...
Silhouette Score: 0.575
Calinski-Harabasz Index: 1976.290
Davies-Bouldin Index: 0.488


### 4. Visualization and Analysis

This section visualizes the clusters and generates summary tables to understand the themes and sentiments within them. The interactive plot allows you to explore each cluster and the reviews it contains.

In [ ]:
MAX_TOOLTIP_LEN = 100
# Create a truncated version of the review text for a cleaner tooltip display in the plot
df_filtered['combined_text_ner_short'] = df_filtered['combined_text_ner'].apply(
    lambda x: x if len(x) <= MAX_TOOLTIP_LEN else x[:MAX_TOOLTIP_LEN] + '...'
)

print("\nCreating interactive cluster visualization...")
# Use Plotly to create an interactive scatter plot of the clusters in 2D space
fig = px.scatter(
    df_filtered[df_filtered['cluster_label'] != -1],
    x='x',
    y='y',
    color='cluster_label',
    color_continuous_scale=px.colors.qualitative.Plotly,
    hover_data={
        'cluster_label': True,
        'entity_text': True,
        'combined_text_ner_short': True,
        'combined_text_ner': False  # Hide the full text from the tooltip for readability
    },
    title=f"Entity-Based Thematic Clusters (HDBSCAN on UMAP) - {n_clusters} Clusters"
)
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(xaxis_title="UMAP 1", yaxis_title="UMAP 2", legend_title_text='Cluster', coloraxis_showscale=False)
fig.show()

# Adjust polarity values (from 1, 2, 3 to 0, 1, 2) for easier sentiment analysis
if df_filtered['polarity'].min() == 1:
    df_filtered['polarity'] = df_filtered['polarity'] - 1
df_clustered = df_filtered[df_filtered['cluster_label'] != -1].copy()

# Function to get the top entities for each cluster using TF-IDF
def get_top_entities_per_cluster(df_group):
    corpus = ' '.join(df_group['entity_text'])
    if not corpus.strip(): return "N/A"
    try:
        vec = TfidfVectorizer(max_features=10, stop_words='english')
        vec.fit_transform([corpus])
        return ', '.join(vec.get_feature_names_out())
    except ValueError: return "N/A"

# Create a summary table for each cluster with the number of reviews and average sentiment
cluster_summary = df_clustered.groupby('cluster_label').agg(
    num_reviews=('cluster_label', 'size'),
    average_sentiment=('polarity', 'mean')
).reset_index()
# Get the principal entities for each cluster and merge with the summary table
top_entities_list = df_clustered.groupby('cluster_label').apply(get_top_entities_per_cluster)
cluster_summary = cluster_summary.merge(top_entities_list.rename('principal_entities'), on='cluster_label')
cluster_summary = cluster_summary.sort_values(by='num_reviews', ascending=False)

print("\nCluster Summary Table (Sentiment and Principal Entities):")
display(cluster_summary.head(20))


Creating interactive cluster visualization...



Cluster Summary Table (Sentiment and Principal Entities):


,cluster_label,num_reviews,average_sentiment,principal_entities
11,11,1720,0.518023,"blade, cannon, chi, dexter, donny, oil, oracle, pampers, quicken, uniden"
31,31,116,0.517241,"clooney, hd, oregon, roc, soc, stephen, stitch, sunforce, teacher, todd"
72,72,60,0.500000,"kt, matthew, night, queen, reserve, secrets, silence, street, tunstall, world"
172,172,56,0.607143,"bbc, clark, good, great, jimmy, life, pat, security, simon, van"
77,77,47,0.340426,"360, america, england, junk, larry, london, microsoft, vista, windows, xbox"
83,83,45,0.511111,"bea, chris, ii, india, java, john, kindle, limbaugh, otis, rush"
81,81,38,0.605263,"america, bellesiles, che, chef, columbia, grisham, hamburger, john, south, university"
136,136,38,0.447368,"angeles, bay, hard, jam, los, pearl, sarah, street, titans, wonder"
162,162,38,0.605263,"bridges, dickens, dunham, enigma, francisco, hardy, jeff, john, mars, san"
103,103,37,0.405405,"better, body, book, bureau, business, charmed, cook, hal, shadows, twilight"


### 5. Sentiment Analysis by Entity Co-occurrence

These final sections analyze the sentiment of reviews based on which entities appear together. This helps us understand if certain combinations of products or people are associated with more positive or negative feedback.

In [ ]:
# Filter the DataFrame to only include reviews with at least one extracted entity
df_analysis = df[df['entities'].apply(lambda x: isinstance(x, list) and len(x) > 0)].copy()

# Create a unique tuple of sorted entities for each review to group by identical entity sets
df_analysis['entity_tuple'] = df_analysis['entities'].apply(lambda entity_list: tuple(sorted(entity_list)))

# Group the reviews by their entity tuple and calculate sentiment statistics
analysis = df_analysis.groupby('entity_tuple')['polarity'].agg(
    Review_Count='size',
    Average_Sentiment=lambda p: round(p.mean(), 2),
    Positive_Reviews=lambda p: (p == 2).sum(),
    Negative_Reviews=lambda p: (p == 1).sum()
).reset_index()

# Convert the entity tuples back into a comma-separated string for display
analysis['Entities'] = analysis['entity_tuple'].apply(lambda x: ', '.join(x))

# Filter for groups with more than one review and sort to find the most common entity sets
report = analysis[analysis['Review_Count'] > 1]
final_report = report.sort_values(by='Review_Count', ascending=False)

print("Sentiment Analysis for Identical Entity Groups")
display(final_report[['Entities', 'Review_Count', 'Average_Sentiment', 'Positive_Reviews', 'Negative_Reviews']])

Sentiment Analysis for Identical Entity Groups


,Entities,Review_Count,Average_Sentiment,Positive_Reviews,Negative_Reviews
5518,sony,19,1.37,7,12
1869,canon,10,1.40,4,6
3873,ipod,10,1.30,3,7
629,america,8,1.38,3,5
5955,walmart,7,1.43,3,4
...,...,...,...,...,...
6007,windows xp,2,2.00,2,0
6019,woodstock,2,1.50,1,1
6028,wwi,2,2.00,2,0
6029,wwii,2,2.00,2,0


### 5.1 Pairwise Entity Analysis

In [ ]:
from itertools import combinations
from collections import defaultdict

# Filter for reviews with at least two entities
df_analysis = df[df['entities'].apply(lambda x: isinstance(x, list) and len(x) >= 2)].copy()

# Dictionaries to store counts and sentiment for each entity pair
pair_counts = defaultdict(int)
pair_sentiments = defaultdict(list)

# Iterate through reviews and extract all unique pairs of entities
for entities_list in df_analysis['entities']:
    unique_entities = sorted(list(set(entities_list)))
    for pair in combinations(unique_entities, 2):
        pair_counts[pair] += 1

pair_sentiments_full = defaultdict(list)
for index, row in df_analysis.iterrows():
    entities_list = row['entities']
    polarity = row['polarity']
    unique_entities = sorted(list(set(entities_list)))
    for pair in combinations(unique_entities, 2):
        pair_sentiments_full[pair].append(polarity)

report_data = []
for pair, sentiments in pair_sentiments_full.items():
    if len(sentiments) > 1:
        report_data.append({
            'Entity_1': pair[0],
            'Entity_2': pair[1],
            'Count': len(sentiments),
            'Average_Sentiment': round(pd.Series(sentiments).mean(), 2)
        })

if report_data:
    final_report = pd.DataFrame(report_data)
    final_report = final_report.sort_values(by='Count', ascending=False)
    print("\nSentiment Analysis for Entity Pairs")
    display(final_report)
else:
    print("No significant pairs found.")


Sentiment Analysis for Entity Pairs


,Entity_1,Entity_2,Count,Average_Sentiment
47,king,stephen king,6,1.50
7,doug,melissa,5,1.40
24,apple,ipod,4,1.25
9,george lucas,star wars,4,1.25
28,message in a bottle,sparks,3,1.33
...,...,...,...,...
123,nora roberts,roberts,2,1.50
124,pinkerton,weezer,2,1.00
125,clay,clay aiken,2,2.00
126,canada,u.s,2,1.00


### 5.2 Triplet Entity Analysis

In [ ]:
# Filter for reviews with at least three entities
df_analysis = df[df['entities'].apply(lambda x: isinstance(x, list) and len(x) >= 3)].copy()

# Dictionaries to store counts and sentiment for each entity triplet
triplet_counts = defaultdict(int)
triplet_sentiments = defaultdict(list)

# Iterate through reviews and extract all unique triplets of entities
for entities_list in df_analysis['entities']:
    unique_entities = sorted(list(set(entities_list)))
    for triplet in combinations(unique_entities, 3):
        triplet_counts[triplet] += 1

triplet_sentiments_full = defaultdict(list)
for index, row in df_analysis.iterrows():
    entities_list = row['entities']
    polarity = row['polarity']
    unique_entities = sorted(list(set(entities_list)))
    for triplet in combinations(unique_entities, 3):
        triplet_sentiments_full[triplet].append(polarity)

report_data = []
for triplet, sentiments in triplet_sentiments_full.items():
    if len(sentiments) >= 1:
        report_data.append({
            'Entity_1': triplet[0],
            'Entity_2': triplet[1],
            'Entity_3': triplet[2],
            'Count': len(sentiments),
            'Average_Sentiment': round(pd.Series(sentiments).mean(), 2)
        })

if report_data:
    final_report = pd.DataFrame(report_data)
    final_report = final_report.sort_values(by='Count', ascending=False)
    print("\nSentiment Analysis for Entity Triplets")
    display(final_report)
else:
    print("No significant triplets found.")


Sentiment Analysis for Entity Triplets


,Entity_1,Entity_2,Entity_3,Count,Average_Sentiment
1238,a walk to remember,sparks,the notebook,2,1.0
20425,abbott,costello,universal,2,2.0
25179,courtney cox,neve campbell,scream,2,1.0
5748,message in a bottle,sparks,the notebook,2,1.0
11888,bush,fahrenheit 911,moore,2,1.5
...,...,...,...,...,...
15616,great 1983 jazz-blues album,quiet storm,workin' couple5,1,2.0
15617,great 1983 jazz-blues album,some rainy day9,toni arden,1,2.0
15618,great 1983 jazz-blues album,some rainy day9,vinnie knight,1,2.0
15619,great 1983 jazz-blues album,some rainy day9,workin' couple5,1,2.0
